In [1]:
import pandas as pd
import numpy as np
import re            #allows bulit in text op(search,split,alter)
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import csr_matrix

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Switch to test set
TEST_DIR = Path("../student_resource/dataset/test")
s1 = pd.read_csv("../student_resource/dataset/train/train_source1.tsv", sep='\t')
s2 = pd.read_csv("../student_resource/dataset/train/train_source2.tsv", sep='\t')
s3 = pd.read_csv("../student_resource/dataset/train/train_source3.tsv", sep='\t')

In [4]:

# Regex patterns for string cleaning
ENTITY_SUFFIXES = [
    # Multi-word & phrase extensions (must be matched FIRST)
    'private limited', 'pvt ltd', 'pvtltd', 'p ltd',
    'limited liability company', 'societe par actions simplifiee',
    'societe a responsabilite limitee', 'auto entrepreneur',
    
    # Standard multi-letter abbreviations
    'incorporated', 'corporation', 'limited', 'company',
    'sasu', 'eurl', 'sarl', 'lllp',
    
    # Common short abbreviations
    'inc', 'llc', 'corp', 'ltd', 'pvt', 'llp', 'plc', 'co', 'pa', 'pc', 'na', 'lp',
    'sas', 'sci', 'snc', 'sa'
]

In [5]:
# Sort by length descending to prevent partial string stripping
sorted_suffixes = sorted(ENTITY_SUFFIXES, key=len, reverse=True)

# Build dynamic regex pattern
escaped_suffixes = [re.escape(s) for s in sorted_suffixes]
SUFFIX_PATTERN = r'\b(' + '|'.join(escaped_suffixes) + r')\b'

def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(SUFFIX_PATTERN, '', text)
    text = re.sub(r'[^a-z0-9\s]', ' ', text)  # Strip special characters
    text = re.sub(r'\s+', ' ', text).strip()
    return text

In [7]:
import pandas as pd
import numpy as np
import re

# Create combined search text (Name + Address)
for df in [s1, s2, s3]:
    # .fillna('').astype(str) guarantees everything is a string before hitting clean_text
    df['clean_name'] = df['business_name'].fillna('').astype(str).apply(clean_text)
    df['clean_address'] = df['business_address'].fillna('').astype(str).apply(clean_text)
    df['search_text'] = df['clean_name'] + " " + df['clean_address']

verification

In [ ]:
# Quick sanity check on 5 random rows
for idx, row in s1.sample(5, random_state=42).iterrows():
    print(f"RAW   : {row['business_name']}")
    print(f"CLEAN : {row['clean_name']}")
    print("-" * 50)

RAW   : Coastal Classic Chiron Inc
CLEAN : coastal classic chiron
--------------------------------------------------
RAW   : Svn Traders LLP
CLEAN : svn traders
--------------------------------------------------
RAW   : École primaire de Georges
CLEAN : cole primaire de georges
--------------------------------------------------
RAW   : Hari Energy Private Limited
CLEAN : hari energy
--------------------------------------------------
RAW   : Marguerite Atelier SARL
CLEAN : marguerite atelier
--------------------------------------------------


In [8]:
from sklearn.neighbors import NearestNeighbors
from tqdm import tqdm

In [9]:
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

def get_knn_candidates(source_df, target_df, k=15, batch_size=2000): # Notice batch_size increased to 2000!
    if source_df.empty or target_df.empty:
        return {}

    # Strict TF-IDF settings (Your winning formula)
    vectorizer = TfidfVectorizer(
        analyzer='word', 
        ngram_range=(1, 2), 
        min_df=5, 
        max_df=0.005, 
        max_features=30000,
        dtype=np.float32, 
        norm='l2'
    )
    
    # 1. Fit & Transform both datasets
    target_tfidf = vectorizer.fit_transform(target_df['search_text'])
    source_tfidf = vectorizer.transform(source_df['search_text'])
    
    # Pre-transpose target matrix for fast C-level matrix multiplication
    target_tfidf_T = target_tfidf.T 
    
    target_ids = target_df['entity_id'].values
    source_ids = source_df['entity_id'].values
    
    candidate_dict = {}
    
    # 2. Chunked Sparse Dot Product
    for start_idx in tqdm(range(0, len(source_ids), batch_size), desc="  Processing Batches"):
        end_idx = min(start_idx + batch_size, len(source_ids))
        
        # This calculates Cosine Similarity directly.
        # Returns a highly sparse matrix of size (batch_size, 1.87M)
        batch_sim = source_tfidf[start_idx:end_idx].dot(target_tfidf_T)
        
        # 3. Extract Top K efficiently without converting to dense
        for i in range(batch_sim.shape[0]):
            s_id = source_ids[start_idx + i]
            
            # Get the non-zero similarities and their column indices (target indices) for this specific source entity
            row_start = batch_sim.indptr[i]
            row_end = batch_sim.indptr[i+1]
            
            if row_start == row_end:
                candidate_dict[s_id] = []
                continue
                
            sims = batch_sim.data[row_start:row_end]
            target_idxs = batch_sim.indices[row_start:row_end]
            
            # Your old threshold: Cosine Distance < 0.85 is equal to Cosine Similarity > 0.15
            valid_mask = sims > 0.15
            sims = sims[valid_mask]
            target_idxs = target_idxs[valid_mask]
            
            if len(sims) == 0:
                candidate_dict[s_id] = []
                continue
            
            # Sort to get Top K highest similarities
            if len(sims) > k:
                # argpartition is wildly faster than np.argsort for large arrays
                top_k_idx = np.argpartition(sims, -k)[-k:]
                # Sort just the top K so they are ordered properly
                top_k_idx = top_k_idx[np.argsort(-sims[top_k_idx])]
            else:
                top_k_idx = np.argsort(-sims)
                
            best_target_idxs = target_idxs[top_k_idx]
            
            # Map back to string entity IDs
            candidate_dict[s_id] = target_ids[best_target_idxs].tolist()
            
    return candidate_dict

In [11]:
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm

def get_knn_candidates(source_df, target_df, k=15, batch_size=1000): 
    if source_df.empty or target_df.empty:
        return {}

    # STRICTER SETTINGS: 
    # min_df=10 ignores rare typos
    # max_df=0.001 aggressively kills common noise causing the slowdown
    vectorizer = TfidfVectorizer(
        analyzer='word', 
        ngram_range=(1, 2), 
        min_df=10, 
        max_df=0.001, 
        max_features=40000,
        dtype=np.float32, 
        norm='l2'
    )
    
    target_tfidf = vectorizer.fit_transform(target_df['search_text'])
    source_tfidf = vectorizer.transform(source_df['search_text'])
    
    target_tfidf_T = target_tfidf.T 
    
    target_ids = target_df['entity_id'].values
    source_ids = source_df['entity_id'].values
    
    candidate_dict = {}
    
    for start_idx in tqdm(range(0, len(source_ids), batch_size), desc="  Processing Batches"):
        end_idx = min(start_idx + batch_size, len(source_ids))
        
        batch_sim = source_tfidf[start_idx:end_idx].dot(target_tfidf_T)
        
        for i in range(batch_sim.shape[0]):
            s_id = source_ids[start_idx + i]
            
            row_start = batch_sim.indptr[i]
            row_end = batch_sim.indptr[i+1]
            
            if row_start == row_end:
                candidate_dict[s_id] = []
                continue
                
            sims = batch_sim.data[row_start:row_end]
            target_idxs = batch_sim.indices[row_start:row_end]
            
            valid_mask = sims > 0.15
            sims = sims[valid_mask]
            target_idxs = target_idxs[valid_mask]
            
            if len(sims) == 0:
                candidate_dict[s_id] = []
                continue
            
            if len(sims) > k:
                top_k_idx = np.argpartition(sims, -k)[-k:]
                top_k_idx = top_k_idx[np.argsort(-sims[top_k_idx])]
            else:
                top_k_idx = np.argsort(-sims)
                
            best_target_idxs = target_idxs[top_k_idx]
            candidate_dict[s_id] = target_ids[best_target_idxs].tolist()
            
    return candidate_dict

In [14]:
import pandas as pd
from pathlib import Path

# 2. Execution Orchestrator
all_candidates = {}
countries = s1['country'].unique()
K_CANDIDATES = 15  

print("\n2. Starting Candidate Generation...")

for country in countries:
    print(f"\nProcessing country: {country}")
    
    s1_sub = s1[s1['country'] == country]
    s2_sub = s2[s2['country'] == country]
    s3_sub = s3[s3['country'] == country]
    
    print(f"  Searching Source 2 ({len(s2_sub):,} records)...")
    s2_cands = get_knn_candidates(s1_sub, s2_sub, k=K_CANDIDATES)
    
    print(f"  Searching Source 3 ({len(s3_sub):,} records)...")
    s3_cands = get_knn_candidates(s1_sub, s3_sub, k=K_CANDIDATES)
    
    for s1_id in s1_sub['entity_id']:
        combined = s2_cands.get(s1_id, []) + s3_cands.get(s1_id, [])
        all_candidates[s1_id] = ",".join(combined)

# Format the dictionary into a DataFrame
candidate_df = pd.DataFrame([
    {"source1_entity_id": k, "candidate_entity_ids": v} 
    for k, v in all_candidates.items()
])

# Ensure 100% of Source 1 IDs are included (even if they have no matches)
candidate_df = pd.merge(
    s1[['entity_id']].rename(columns={'entity_id': 'source1_entity_id'}), 
    candidate_df, 
    on='source1_entity_id', 
    how='left'
).fillna("")

# Save directly as the train file
output_dir = Path("../output")
output_dir.mkdir(parents=True, exist_ok=True)
output_file = output_dir / "candidate_pairs_train.tsv"

candidate_df.to_csv(output_file, sep="\t", index=False)
print(f"\nFinished! Saved {len(candidate_df):,} rows to {output_file}")


2. Starting Candidate Generation...

Processing country: US
  Searching Source 2 (3,016,817 records)...


  Processing Batches: 100%|██████████| 1324/1324 [16:25<00:00,  1.34it/s]


  Searching Source 3 (3,170,056 records)...


  Processing Batches: 100%|██████████| 1324/1324 [13:18<00:00,  1.66it/s]



Processing country: India
  Searching Source 2 (2,017,799 records)...


  Processing Batches: 100%|██████████| 884/884 [04:37<00:00,  3.19it/s]


  Searching Source 3 (2,115,547 records)...


  Processing Batches: 100%|██████████| 884/884 [04:52<00:00,  3.02it/s]



Finished! Saved 2,206,821 rows to ..\output\candidate_pairs_train.tsv
